In [4]:
# SENTINEL 8.3 — Quant Engine (CASH MODE, Price/Trend only; IA se integra después)
# Archivo integrado — Copiar/pegar tal cual.

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, List, Tuple


@dataclass
class CostsConfig:
    commission: float = 0.001
    slippage: float = 0.0002
    swap_daily: float = 0.0003
    apply_slippage: bool = True
    apply_swap: bool = False  # CASH MODE


@dataclass
class BacktestConfig:
    capital_initial: float = 100000.0
    start: str = "2006-01-01"
    end: str = "2026-02-20"
    burn_in: int = 200
    trading_days: int = 252
    rf_annual: float = 0.0
    rebalance_mode: str = "event"   # "event" recomendado
    rebalance_freq: str = "W-FRI"   # si "calendar"


@dataclass
class CVConfig:
    train_years: int = 8
    test_years: int = 1
    purge_days: int = 10
    embargo_days: int = 5


@dataclass
class ScoreConfig:
    dd_cap: float = -0.35
    lambda_dd: float = 2.0
    lambda_var: float = 0.5
    min_splits: int = 5


ESCENARIOS: Dict[str, List[str]] = {
    "tech_original": ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'AVGO', 'ASML', 'TSM', 'ADBE', 'NFLX', 'AMD'],
    "tech_alternativo": ['CRM', 'ORCL', 'IBM', 'INTC', 'CSCO', 'QCOM', 'TXN', 'ADI', 'MU', 'ANET', 'PLTR', 'SNPS'],
    "defensivo": ['JNJ', 'PG', 'KO', 'PEP', 'WMT', 'COST', 'MO', 'CL', 'GIS', 'KMB', 'MDLZ', 'CPB'],
    "industrial": ['HON', 'CAT', 'DE', 'BA', 'LMT', 'GE', 'MMM', 'EMR', 'ITW', 'ETN', 'ROK', 'DOV'],
    "aleatorio": ['AAPL', 'MSFT', 'JNJ', 'PG', 'CAT', 'XOM', 'CVX', 'JPM', 'GS', 'V', 'MA', 'KO'],
}

BENCH_NASDAQ100 = "QQQ"
BENCH_SP500 = "SPY"

STRESS_EPISODES = {
    "GFC_2008": ("2008-01-01", "2009-06-30"),
    "EURO_2011": ("2011-07-01", "2011-12-31"),
    "Q4_2018": ("2018-10-01", "2018-12-31"),
    "COVID_2020": ("2020-02-15", "2020-06-30"),
    "RATES_2022": ("2022-01-01", "2022-12-31"),
}


def to_1d_series(x, name: str = "x") -> pd.Series:
    if x is None:
        return pd.Series(dtype=float, name=name)
    if isinstance(x, pd.Series):
        return x.rename(name)
    if isinstance(x, pd.DataFrame):
        if x.shape[1] == 0:
            return pd.Series(dtype=float, name=name)
        return x.iloc[:, 0].rename(name)
    arr = np.asarray(x)
    if arr.ndim == 2 and arr.shape[1] == 1:
        arr = arr.reshape(-1)
    return pd.Series(arr, name=name)


def kama(price: np.ndarray, n: int = 10, fast: int = 2, slow: int = 30) -> np.ndarray:
    price = np.asarray(price, dtype=float)
    length = len(price)
    out = np.full(length, np.nan)
    if length == 0:
        return out

    out[0] = price[0]
    fast_sc = 2.0 / (fast + 1.0)
    slow_sc = 2.0 / (slow + 1.0)

    for i in range(1, length):
        if i < n:
            out[i] = price[i]
        else:
            change = abs(price[i] - price[i - n])
            volatility = np.sum(np.abs(np.diff(price[i - n + 1:i + 1])))
            er = 0.0 if volatility == 0 else change / volatility
            sc = (er * (fast_sc - slow_sc) + slow_sc) ** 2
            out[i] = out[i - 1] + sc * (price[i] - out[i - 1])
    return out


def download_prices(tickers: List[str], start: str, end: str) -> pd.DataFrame:
    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, group_by="ticker", progress=False)
    idx = pd.bdate_range(start=start, end=end)
    df = pd.DataFrame(index=idx)

    for t in tickers:
        try:
            if isinstance(raw, pd.DataFrame) and "Close" in raw.columns:
                s = raw["Close"]
                if isinstance(s, pd.DataFrame):
                    s = s[t] if t in s.columns else None
            else:
                s = raw[t]["Close"]

            if s is None:
                continue

            s = pd.Series(s).dropna()
            if s.empty:
                continue

            s = s.reindex(idx)
            s = s.ffill(limit=5)
            df[t] = s
        except Exception:
            continue

    return df.dropna(how="all")


def download_benchmark(ticker: str, start: str, end: str) -> pd.Series:
    data = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if isinstance(data, pd.DataFrame):
        if "Close" in data.columns:
            s = data["Close"]
        elif "Adj Close" in data.columns:
            s = data["Adj Close"]
        else:
            s = data.select_dtypes(include=[np.number]).iloc[:, 0]
    else:
        s = data
    return to_1d_series(s, name=ticker).dropna()


def align_returns_no_fill(price_series, idx: pd.DatetimeIndex) -> pd.Series:
    p = to_1d_series(price_series, "bench_price").reindex(idx)
    return p.pct_change()


def compute_positions_kama(prices: pd.DataFrame, params: Dict, burn_in: int) -> pd.DataFrame:
    n, fast, slow = params["n"], params["fast"], params["slow"]
    pos = pd.DataFrame(index=prices.index, columns=prices.columns, dtype=float)

    for t in prices.columns:
        p = prices[t]
        if p.isna().all():
            pos[t] = 0.0
            continue

        k = pd.Series(kama(p.values, n=n, fast=fast, slow=slow), index=p.index).shift(1)
        sig = ((p > k) & p.notna() & k.notna()).astype(float)
        sig.iloc[:burn_in] = 0.0
        pos[t] = sig.fillna(0.0)

    return pos.fillna(0.0)


def event_rebalance_weights(pos: pd.DataFrame) -> pd.DataFrame:
    active = (pos > 0).astype(float)
    n_active = active.sum(axis=1).replace(0, np.nan)
    return active.div(n_active, axis=0).fillna(0.0)


def calendar_rebalance_weights(pos: pd.DataFrame, freq: str = "W-FRI") -> pd.DataFrame:
    w_event = event_rebalance_weights(pos)
    reb_dates = w_event.resample(freq).last().index
    w = pd.DataFrame(index=w_event.index, columns=w_event.columns, dtype=float)

    last_w = np.zeros(len(w_event.columns))
    for dt in w_event.index:
        if dt in reb_dates:
            last_w = w_event.loc[dt].values
        w.loc[dt] = last_w

    return w.fillna(0.0)


def compute_turnover(weights: pd.DataFrame) -> pd.Series:
    dw = weights.diff().abs().fillna(0.0)
    return 0.5 * dw.sum(axis=1)


def apply_costs(port_gross: pd.Series, weights: pd.DataFrame, costs: CostsConfig) -> pd.Series:
    to = compute_turnover(weights)
    trade_cost = to * (costs.commission + (costs.slippage if costs.apply_slippage else 0.0))
    exposure = weights.abs().sum(axis=1)
    swap_cost = exposure * (costs.swap_daily if costs.apply_swap else 0.0)
    return port_gross - trade_cost - swap_cost


def backtest_portfolio(prices: pd.DataFrame, params: Dict, bt_cfg: BacktestConfig, costs: CostsConfig) -> Dict:
    pos = compute_positions_kama(prices, params, bt_cfg.burn_in)

    if bt_cfg.rebalance_mode == "calendar":
        w = calendar_rebalance_weights(pos, freq=bt_cfg.rebalance_freq)
    else:
        w = event_rebalance_weights(pos)

    rets = prices.pct_change()
    port_gross = (w.shift(1).fillna(0.0) * rets.fillna(0.0)).sum(axis=1)
    port_net = apply_costs(port_gross, w, costs).replace([np.inf, -np.inf], 0.0).fillna(0.0)

    equity = bt_cfg.capital_initial * (1.0 + port_net).cumprod()
    exposure = w.abs().sum(axis=1)

    return {
        "returns_gross": port_gross,
        "returns_net": port_net,
        "equity": equity,
        "positions": pos,
        "weights": w,
        "turnover": compute_turnover(w),
        "exposure": exposure,
    }


def buy_and_hold_equal_weight(prices: pd.DataFrame, bt_cfg: BacktestConfig) -> Dict:
    first = prices.iloc[0]
    cols = [c for c in prices.columns if not pd.isna(first[c])]
    if len(cols) == 0:
        cols = [c for c in prices.columns if prices[c].notna().any()]

    sub = prices[cols].copy()
    w0 = pd.Series(1.0 / len(cols), index=cols)

    rets = sub.pct_change().fillna(0.0)
    port = (rets * w0).sum(axis=1)

    equity = bt_cfg.capital_initial * (1.0 + port).cumprod()
    weights = pd.DataFrame(index=sub.index, columns=sub.columns, data=np.tile(w0.values, (len(sub), 1)))
    turnover = compute_turnover(weights)
    exposure = weights.abs().sum(axis=1)

    return {
        "returns_net": port,
        "equity": equity,
        "weights": weights,
        "turnover": turnover,
        "exposure": exposure,
        "tickers_used": cols,
    }


def total_return(ret: pd.Series) -> float:
    r = to_1d_series(ret, "r").dropna()
    if len(r) == 0:
        return 0.0
    return float((1.0 + r).prod() - 1.0)


def cagr(ret: pd.Series, trading_days: int = 252) -> float:
    r = to_1d_series(ret, "r").dropna()
    if len(r) == 0:
        return 0.0
    tr = (1.0 + r).prod() - 1.0
    return float((1.0 + tr) ** (trading_days / len(r)) - 1.0)


def ann_vol(ret: pd.Series, trading_days: int = 252) -> float:
    r = to_1d_series(ret, "r").dropna()
    sd = r.std(ddof=1)
    return float(sd * np.sqrt(trading_days)) if sd == sd else np.nan


def sharpe_ratio(ret: pd.Series, rf_annual: float = 0.0, trading_days: int = 252) -> float:
    r = to_1d_series(ret, "r").dropna()
    rf_daily = (1.0 + rf_annual) ** (1.0 / trading_days) - 1.0
    ex = r - rf_daily
    sd = ex.std(ddof=1)
    if sd == 0 or np.isnan(sd):
        return 0.0
    return float(np.sqrt(trading_days) * ex.mean() / sd)


def sortino_ratio(ret: pd.Series, rf_annual: float = 0.0, trading_days: int = 252) -> float:
    r = to_1d_series(ret, "r").dropna()
    rf_daily = (1.0 + rf_annual) ** (1.0 / trading_days) - 1.0
    ex = r - rf_daily
    downside = ex.copy()
    downside[downside > 0] = 0
    dd = downside.std(ddof=1)
    if dd == 0 or np.isnan(dd):
        return 0.0
    return float(np.sqrt(trading_days) * ex.mean() / dd)


def max_drawdown(equity: pd.Series) -> float:
    eq = to_1d_series(equity, "eq").dropna()
    if len(eq) == 0:
        return 0.0
    peak = eq.cummax()
    dd = eq / peak - 1.0
    return float(dd.min())


def calmar_ratio(ret: pd.Series, equity: pd.Series, trading_days: int = 252) -> float:
    a = cagr(ret, trading_days)
    dd = max_drawdown(equity)
    return float(a / abs(dd)) if dd != 0 else np.inf


def cvar(ret: pd.Series, alpha: float = 0.05) -> float:
    r = to_1d_series(ret, "r").dropna().values
    if len(r) == 0:
        return np.nan
    q = np.quantile(r, alpha)
    tail = r[r <= q]
    return float(tail.mean()) if len(tail) else float(q)


def summarize_bt(bt: Dict, bt_cfg: BacktestConfig) -> Dict:
    r = to_1d_series(bt["returns_net"], "r").replace([np.inf, -np.inf], np.nan).dropna()
    eq = to_1d_series(bt["equity"], "eq").dropna()

    summary = {
        "FinalEquity": float(eq.iloc[-1]) if len(eq) else np.nan,
        "TotalReturn": total_return(r),
        "CAGR": cagr(r, bt_cfg.trading_days),
        "AnnVol": ann_vol(r, bt_cfg.trading_days),
        "Sharpe": sharpe_ratio(r, bt_cfg.rf_annual, bt_cfg.trading_days),
        "Sortino": sortino_ratio(r, bt_cfg.rf_annual, bt_cfg.trading_days),
        "MaxDD": max_drawdown(eq),
        "Calmar": calmar_ratio(r, eq, bt_cfg.trading_days),
        "CVaR_5": cvar(r, 0.05),
        "CVaR_1": cvar(r, 0.01),
        "Days": int(len(r)),
    }

    to = to_1d_series(bt.get("turnover", pd.Series(0, index=r.index)), "to").reindex(r.index).fillna(0.0)
    summary["TurnoverAnn"] = float(to.sum() * (bt_cfg.trading_days / len(r))) if len(r) else 0.0

    exp = to_1d_series(bt.get("exposure", pd.Series(0, index=r.index)), "exp").reindex(r.index).fillna(0.0)
    summary["AvgExposure"] = float(exp.mean()) if len(exp) else np.nan
    summary["TimeInMarket"] = float((exp > 0).mean()) if len(exp) else np.nan

    return summary


def newey_west_se(X: np.ndarray, resid: np.ndarray, lags: int = 5) -> np.ndarray:
    T, k = X.shape
    S = np.zeros((k, k))

    for t in range(T):
        xt = X[t:t+1].T
        S += (resid[t] ** 2) * (xt @ xt.T)

    for L in range(1, lags + 1):
        w = 1.0 - L / (lags + 1.0)
        Gamma = np.zeros((k, k))
        for t in range(L, T):
            xt = X[t:t+1].T
            xtL = X[t-L:t-L+1].T
            Gamma += resid[t] * resid[t-L] * (xt @ xtL.T)
        S += w * (Gamma + Gamma.T)

    XtX_inv = np.linalg.inv(X.T @ X)
    V = XtX_inv @ S @ XtX_inv
    return np.sqrt(np.diag(V))


def alpha_beta_nw(ret_strategy, ret_mkt, rf_annual: float = 0.0, trading_days: int = 252, lags: int = 5) -> Dict:
    s = to_1d_series(ret_strategy, "strategy")
    m = to_1d_series(ret_mkt, "market")

    rf_daily = (1.0 + rf_annual) ** (1.0 / trading_days) - 1.0
    df = pd.DataFrame({"s": s, "m": m}).dropna()
    if len(df) < 50:
        return {"alpha_ann": np.nan, "beta": np.nan, "t_alpha": np.nan}

    y = (df["s"] - rf_daily).values
    x1 = (df["m"] - rf_daily).values
    X = np.column_stack([np.ones(len(df)), x1])

    b = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - X @ b

    se = newey_west_se(X, resid, lags=lags)
    t_alpha = b[0] / se[0] if se[0] != 0 else np.nan

    alpha_daily = b[0]
    alpha_ann = (1.0 + alpha_daily) ** trading_days - 1.0

    return {"alpha_ann": float(alpha_ann), "beta": float(b[1]), "t_alpha": float(t_alpha)}


def purged_walk_forward_splits(index: pd.DatetimeIndex, cv_cfg: CVConfig) -> List[Tuple[pd.DatetimeIndex, pd.DatetimeIndex]]:
    years = sorted(index.year.unique())
    splits = []

    for i in range(cv_cfg.train_years, len(years) - cv_cfg.test_years + 1):
        train_years = years[i - cv_cfg.train_years:i]
        test_years = years[i:i + cv_cfg.test_years]

        train_idx = index[index.year.isin(train_years)]
        test_idx = index[index.year.isin(test_years)]
        if len(train_idx) == 0 or len(test_idx) == 0:
            continue

        test_start_real = test_idx.min()

        if len(train_idx) > cv_cfg.purge_days:
            train_idx = train_idx[:-cv_cfg.purge_days]
        if len(test_idx) > cv_cfg.purge_days:
            test_idx = test_idx[cv_cfg.purge_days:]

        if len(train_idx) == 0 or len(test_idx) == 0:
            continue

        embargo_cut = test_start_real - pd.tseries.offsets.BDay(cv_cfg.embargo_days)
        train_idx = train_idx[train_idx <= embargo_cut]

        if len(train_idx) < 200 or len(test_idx) < 100:
            continue

        splits.append((train_idx, test_idx))

    return splits


def score_params_cv(prices: pd.DataFrame, params: Dict, bt_cfg: BacktestConfig,
                    costs: CostsConfig, cv_cfg: CVConfig, sc_cfg: ScoreConfig) -> Dict:
    splits = purged_walk_forward_splits(prices.index, cv_cfg)
    if len(splits) < sc_cfg.min_splits:
        return {"score": -np.inf, "avg_sharpe": np.nan, "std_sharpe": np.nan,
                "worst_dd": np.nan, "avg_calmar": np.nan, "splits": len(splits)}

    sharpes, calmars, dds = [], [], []
    for _, test_idx in splits:
        bt = backtest_portfolio(prices.loc[test_idx], params, bt_cfg, costs)
        s = summarize_bt(bt, bt_cfg)
        sharpes.append(s["Sharpe"])
        calmars.append(s["Calmar"])
        dds.append(s["MaxDD"])

    avg_sh = float(np.nanmean(sharpes))
    std_sh = float(np.nanstd(sharpes))
    avg_ca = float(np.nanmean(calmars))
    worst_dd = float(np.nanmin(dds))

    dd_excess = max(0.0, abs(worst_dd) - abs(sc_cfg.dd_cap))
    score = avg_sh - sc_cfg.lambda_dd * dd_excess - sc_cfg.lambda_var * std_sh

    return {
        "score": float(score),
        "avg_sharpe": avg_sh,
        "std_sharpe": std_sh,
        "avg_calmar": avg_ca,
        "worst_dd": worst_dd,
        "splits": len(splits),
    }


def grid_search_kama(prices: pd.DataFrame, bt_cfg: BacktestConfig, costs: CostsConfig,
                     cv_cfg: CVConfig, sc_cfg: ScoreConfig,
                     space_n: List[int], space_fast: List[int], space_slow: List[int]) -> Dict:
    best = {"score": -np.inf, "params": None, "report": None}
    rows = []

    for n in space_n:
        for fast in space_fast:
            for slow in space_slow:
                if fast >= slow:
                    continue
                params = {"n": n, "fast": fast, "slow": slow}
                rep = score_params_cv(prices, params, bt_cfg, costs, cv_cfg, sc_cfg)
                rows.append({**params, **rep})
                if rep["score"] > best["score"]:
                    best = {"score": rep["score"], "params": params, "report": rep}

    df = pd.DataFrame(rows).sort_values("score", ascending=False)
    return {"best": best, "table": df}


def evaluate_stress_episodes(prices: pd.DataFrame, params: Dict, bt_cfg: BacktestConfig, costs: CostsConfig,
                             episodes: Dict[str, Tuple[str, str]], burn_in_episode: int = 0) -> pd.DataFrame:
    out = []
    for name, (a, b) in episodes.items():
        sub = prices.loc[a:b]
        if len(sub) < 40:
            continue

        bt_cfg_ep = BacktestConfig(**{**bt_cfg.__dict__, "burn_in": burn_in_episode})
        bt = backtest_portfolio(sub, params, bt_cfg_ep, costs)
        s = summarize_bt(bt, bt_cfg_ep)

        out.append({
            "Episode": name,
            "Start": a,
            "End": b,
            "CAGR%": round(s["CAGR"] * 100, 2),
            "Sharpe": round(s["Sharpe"], 3),
            "MaxDD%": round(s["MaxDD"] * 100, 2),
            "Calmar": round(s["Calmar"], 3),
            "TurnoverAnn": round(s["TurnoverAnn"], 2),
            "AvgExposure%": round(s["AvgExposure"] * 100, 1),
            "TimeInMkt%": round(s["TimeInMarket"] * 100, 1),
        })

    return pd.DataFrame(out)


def moving_block_bootstrap(series: pd.Series, block: int = 20, n_samples: int = 500, seed: int = 42) -> Dict:
    rng = np.random.default_rng(seed)
    x = to_1d_series(series, "x").replace([np.inf, -np.inf], np.nan).dropna().values
    T = len(x)
    if T < block * 5:
        return {"dd_p50": np.nan, "dd_p95": np.nan, "ruin_prob_50dd": np.nan}

    dds = []
    for _ in range(n_samples):
        starts = rng.integers(0, T - block, size=int(np.ceil(T / block)))
        sample = np.concatenate([x[s:s + block] for s in starts])[:T]
        eq = (1.0 + sample).cumprod()
        peak = np.maximum.accumulate(eq)
        dd = np.min(eq / peak - 1.0)
        dds.append(dd)

    dds = np.array(dds)
    return {
        "dd_p50": float(np.quantile(dds, 0.50)),
        "dd_p95": float(np.quantile(dds, 0.05)),
        "ruin_prob_50dd": float(np.mean(dds < -0.5)) * 100.0
    }


def print_top_params(cv_table: pd.DataFrame, top: int = 15):
    cols = ["n", "fast", "slow", "score", "avg_sharpe", "std_sharpe", "worst_dd", "avg_calmar", "splits"]
    view = cv_table[cols].head(top).copy()
    view["worst_dd"] = (view["worst_dd"] * 100).round(2)
    print("\nTOP PARAMS (CV robust score):")
    print(view.to_string(index=False))


def build_comparison_df(summaries: Dict[str, Dict]) -> pd.DataFrame:
    order = ["SENTINEL", "BUY_HOLD", BENCH_NASDAQ100, BENCH_SP500]
    rows = []
    for k in order:
        if k not in summaries:
            continue
        s = summaries[k]
        rows.append({
            "Model": k,
            "FinalEquity": round(s["FinalEquity"], 2),
            "TotalReturn%": round(s["TotalReturn"] * 100, 2),
            "CAGR%": round(s["CAGR"] * 100, 2),
            "AnnVol%": round(s["AnnVol"] * 100, 2) if s["AnnVol"] == s["AnnVol"] else np.nan,
            "Sharpe": round(s["Sharpe"], 3),
            "Sortino": round(s["Sortino"], 3),
            "MaxDD%": round(s["MaxDD"] * 100, 2),
            "Calmar": round(s["Calmar"], 3),
            "TurnoverAnn": round(s["TurnoverAnn"], 2),
            "AvgExposure%": round(s["AvgExposure"] * 100, 1),
            "TimeInMkt%": round(s["TimeInMarket"] * 100, 1),
            "CVaR5%": round(s["CVaR_5"] * 100, 3),
            "CVaR1%": round(s["CVaR_1"] * 100, 3),
        })
    return pd.DataFrame(rows)


def print_comparison(label: str, summaries: Dict[str, Dict], ab_q: Dict, ab_sp: Dict, boot: Dict, stress: pd.DataFrame):
    df = build_comparison_df(summaries)
    print("\n" + "=" * 140)
    print(f"COMPARATIVO PRINCIPAL — {label}")
    print("=" * 140)
    print(df.to_string(index=False))

    print("\nALFA/BETA (OLS + Newey–West)")
    print(f"Vs {BENCH_NASDAQ100}: alpha_ann={ab_q['alpha_ann']*100:,.2f}% | beta={ab_q['beta']:.3f} | t(alpha)={ab_q['t_alpha']:.3f}")
    print(f"Vs {BENCH_SP500}:   alpha_ann={ab_sp['alpha_ann']*100:,.2f}% | beta={ab_sp['beta']:.3f} | t(alpha)={ab_sp['t_alpha']:.3f}")

    print("\nSIMULACIÓN (Moving Block Bootstrap)")
    print(f"DD p50: {boot['dd_p50']*100:,.2f}% | DD p95(worst 5%): {boot['dd_p95']*100:,.2f}% | P(DD<-50%): {boot['ruin_prob_50dd']:,.2f}%")

    if stress is not None and not stress.empty:
        print("\nSTRESS TESTS (episodios reales; burn_in_episode=0)")
        print(stress.to_string(index=False))


def plot_equity_curves(label: str, curves: Dict[str, pd.Series]):
    plt.figure()
    for k, eq in curves.items():
        eq = to_1d_series(eq, k).dropna()
        if len(eq) == 0:
            continue
        plt.plot(eq.index, eq.values, label=k)
    plt.title(f"Equity Curves — {label}")
    plt.xlabel("Date")
    plt.ylabel("Equity")
    plt.legend()
    plt.show()


def plot_drawdown(label: str, equity: pd.Series):
    eq = to_1d_series(equity, "eq").dropna()
    dd = eq / eq.cummax() - 1.0
    plt.figure()
    plt.plot(dd.index, dd.values)
    plt.title(f"Drawdown — {label}")
    plt.xlabel("Date")
    plt.ylabel("Drawdown")
    plt.show()


def plot_rolling(label: str, ret: pd.Series, window: int = 126):
    r = to_1d_series(ret, "r").replace([np.inf, -np.inf], np.nan).dropna()
    if len(r) < window * 2:
        print(f"[WARN] Pocos datos para rolling window={window}.")
        return
    roll_std = r.rolling(window).std()
    roll_sh = (r.rolling(window).mean() / roll_std).replace([np.inf, -np.inf], np.nan) * np.sqrt(252)
    roll_vol = roll_std * np.sqrt(252)

    plt.figure()
    plt.plot(roll_sh.index, roll_sh.values)
    plt.title(f"Rolling Sharpe ({window}d) — {label}")
    plt.xlabel("Date")
    plt.ylabel("Sharpe")
    plt.show()

    plt.figure()
    plt.plot(roll_vol.index, roll_vol.values)
    plt.title(f"Rolling Volatility ({window}d) — {label}")
    plt.xlabel("Date")
    plt.ylabel("Ann. Vol")
    plt.show()


def plot_exposure_turnover(label: str, exposure: pd.Series, turnover: pd.Series):
    exp = to_1d_series(exposure, "exp").fillna(0.0)
    to = to_1d_series(turnover, "to").fillna(0.0)

    plt.figure()
    plt.plot(exp.index, exp.values)
    plt.title(f"Exposure — {label}")
    plt.xlabel("Date")
    plt.ylabel("Gross exposure")
    plt.show()

    plt.figure()
    plt.plot(to.index, to.values)
    plt.title(f"Turnover (daily) — {label}")
    plt.xlabel("Date")
    plt.ylabel("Turnover")
    plt.show()


def run_sentinel83_single(
    scenario_name: str,
    tickers: List[str],
    bt_cfg: BacktestConfig,
    costs: CostsConfig,
    cv_cfg: CVConfig,
    sc_cfg: ScoreConfig,
    space_n: List[int],
    space_fast: List[int],
    space_slow: List[int],
    make_plots: bool = True,
    top_params_to_print: int = 15,
) -> Dict:

    print("=" * 140)
    print(f"SENTINEL 8.3 — ESCENARIO: {scenario_name} | tickers={len(tickers)} | CASH MODE (swap OFF)")
    print(f"Periodo: {bt_cfg.start} -> {bt_cfg.end} | Rebalance: {bt_cfg.rebalance_mode} | CV DD cap: {sc_cfg.dd_cap:.0%}")
    print("=" * 140)

    prices = download_prices(tickers, bt_cfg.start, bt_cfg.end)
    if prices.empty or prices.shape[1] < 2:
        raise RuntimeError("Datos insuficientes. Revise tickers/fechas o disponibilidad en yfinance.")

    print("\n[1] Grid Search (Purged WFCV + embargo; score robusto)...")
    search = grid_search_kama(prices, bt_cfg, costs, cv_cfg, sc_cfg, space_n, space_fast, space_slow)
    best = search["best"]
    cv_table = search["table"]
    best_params = best["params"]

    print(f"Mejores params: {best_params} | score={best['score']:.3f} | avgSharpe={best['report']['avg_sharpe']:.3f} "
          f"| stdSharpe={best['report']['std_sharpe']:.3f} | worstDD(CV)={best['report']['worst_dd']:.2%} | splits={best['report']['splits']}")
    print_top_params(cv_table, top=top_params_to_print)

    print("\n[2] Backtest Sentinel (full-period, net, cash)...")
    bt_s = backtest_portfolio(prices, best_params, bt_cfg, costs)
    sum_s = summarize_bt(bt_s, bt_cfg)

    print("\n[3] Buy & Hold equiponderado (mismo universo)...")
    bt_bh = buy_and_hold_equal_weight(prices, bt_cfg)
    sum_bh = summarize_bt(bt_bh, bt_cfg)

    print("\n[4] Benchmarks: QQQ (Nasdaq-100) y SPY (S&P 500)...")
    qqq_r = align_returns_no_fill(download_benchmark(BENCH_NASDAQ100, bt_cfg.start, bt_cfg.end), prices.index)
    spy_r = align_returns_no_fill(download_benchmark(BENCH_SP500, bt_cfg.start, bt_cfg.end), prices.index)

    qqq_r = to_1d_series(qqq_r, "QQQ").replace([np.inf, -np.inf], np.nan).dropna()
    spy_r = to_1d_series(spy_r, "SPY").replace([np.inf, -np.inf], np.nan).dropna()

    qqq_eq = bt_cfg.capital_initial * (1.0 + qqq_r).cumprod()
    spy_eq = bt_cfg.capital_initial * (1.0 + spy_r).cumprod()

    sum_q = summarize_bt({"returns_net": qqq_r, "equity": qqq_eq, "turnover": pd.Series(0, index=qqq_r.index),
                          "exposure": pd.Series(1, index=qqq_r.index)}, bt_cfg)
    sum_sp = summarize_bt({"returns_net": spy_r, "equity": spy_eq, "turnover": pd.Series(0, index=spy_r.index),
                           "exposure": pd.Series(1, index=spy_r.index)}, bt_cfg)

    ab_q = alpha_beta_nw(bt_s["returns_net"], qqq_r, bt_cfg.rf_annual, bt_cfg.trading_days, lags=5)
    ab_sp = alpha_beta_nw(bt_s["returns_net"], spy_r, bt_cfg.rf_annual, bt_cfg.trading_days, lags=5)

    print("\n[5] Stress tests (episodios reales; burn_in_episode=0)...")
    stress = evaluate_stress_episodes(prices, best_params, bt_cfg, costs, STRESS_EPISODES, burn_in_episode=0)

    print("\n[6] Simulación: Moving Block Bootstrap (DD y ruina)...")
    boot = moving_block_bootstrap(bt_s["returns_net"], block=20, n_samples=500, seed=42)

    summaries = {
        "SENTINEL": sum_s,
        "BUY_HOLD": sum_bh,
        BENCH_NASDAQ100: sum_q,
        BENCH_SP500: sum_sp,
    }
    print_comparison(scenario_name, summaries, ab_q, ab_sp, boot, stress)

    if make_plots:
        label = f"{scenario_name} | params={best_params}"
        curves = {
            "SENTINEL": bt_s["equity"],
            "BUY_HOLD": bt_bh["equity"],
            BENCH_NASDAQ100: qqq_eq,
            BENCH_SP500: spy_eq,
        }
        plot_equity_curves(label, curves)
        plot_drawdown(label, bt_s["equity"])
        plot_rolling(label, bt_s["returns_net"], window=126)
        plot_exposure_turnover(label, bt_s["exposure"], bt_s["turnover"])

    comp_df = build_comparison_df(summaries)
    comp_df.to_csv(f"sentinel83_{scenario_name}_comparison.csv", index=False)
    stress.to_csv(f"sentinel83_{scenario_name}_stress.csv", index=False)
    cv_table.head(50).to_csv(f"sentinel83_{scenario_name}_cv_top50.csv", index=False)

    print(f"\n[INFO] Exportados CSV: sentinel83_{scenario_name}_comparison.csv, sentinel83_{scenario_name}_stress.csv, sentinel83_{scenario_name}_cv_top50.csv")

    return {
        "scenario": scenario_name,
        "tickers_used": list(prices.columns),
        "best_params": best_params,
        "cv_table": cv_table,
        "best_cv_report": best["report"],
        "sentinel": bt_s,
        "buy_hold": bt_bh,
        "bench": {"QQQ_returns": qqq_r, "SPY_returns": spy_r, "QQQ_equity": qqq_eq, "SPY_equity": spy_eq},
        "summaries": summaries,
        "alpha_beta": {"vs_QQQ": ab_q, "vs_SPY": ab_sp},
        "stress": stress,
        "bootstrap": boot,
        "comparison_df": comp_df,
    }


def run_sentinel83_multicontext(make_plots_each: bool = False) -> List[Dict]:
    bt_cfg = BacktestConfig(
        capital_initial=100000,
        start="2006-01-01",
        end="2026-02-20",
        burn_in=200,
        rf_annual=0.0,
        rebalance_mode="event",
        rebalance_freq="W-FRI"
    )

    costs = CostsConfig(
        commission=0.001,
        slippage=0.0002,
        swap_daily=0.0003,
        apply_slippage=True,
        apply_swap=False
    )

    cv_cfg = CVConfig(train_years=8, test_years=1, purge_days=10, embargo_days=5)
    sc_cfg = ScoreConfig(dd_cap=-0.35, lambda_dd=2.0, lambda_var=0.5, min_splits=5)

    space_n = [20, 30, 40, 50, 60, 80, 100]
    space_fast = [2, 3, 5]
    space_slow = [30, 40, 50, 60, 80, 100]

    results = []
    for name, tickers in ESCENARIOS.items():
        try:
            res = run_sentinel83_single(
                scenario_name=name,
                tickers=tickers,
                bt_cfg=bt_cfg,
                costs=costs,
                cv_cfg=cv_cfg,
                sc_cfg=sc_cfg,
                space_n=space_n,
                space_fast=space_fast,
                space_slow=space_slow,
                make_plots=make_plots_each,
                top_params_to_print=10
            )
            results.append(res)
        except Exception as e:
            print(f"[ERROR] Escenario {name}: {e}")

    return results


if __name__ == "__main__":
    # Multicontexto sin plots (por defecto)
    _ = run_sentinel83_multicontext(make_plots_each=False)

    # Para correr solo el escenario clave con plots:
    # _ = run_sentinel83_single(
    #     scenario_name="tech_original",
    #     tickers=ESCENARIOS["tech_original"],
    #     bt_cfg=BacktestConfig(rebalance_mode="event"),
    #     costs=CostsConfig(apply_swap=False),
    #     cv_cfg=CVConfig(),
    #     sc_cfg=ScoreConfig(dd_cap=-0.35),
    #     space_n=[20, 30, 40, 50, 60, 80, 100],
    #     space_fast=[2, 3, 5],
    #     space_slow=[30, 40, 50, 60, 80, 100],
    #     make_plots=True
    # )

SENTINEL 8.3 — ESCENARIO: tech_original | tickers=12 | CASH MODE (swap OFF)
Periodo: 2006-01-01 -> 2026-02-20 | Rebalance: event | CV DD cap: -35%

[1] Grid Search (Purged WFCV + embargo; score robusto)...
Mejores params: {'n': 50, 'fast': 5, 'slow': 40} | score=0.480 | avgSharpe=0.906 | stdSharpe=0.852 | worstDD(CV)=-17.53% | splits=12

TOP PARAMS (CV robust score):
 n  fast  slow    score  avg_sharpe  std_sharpe  worst_dd  avg_calmar  splits
50     5    40 0.480359    0.906271    0.851824    -17.53    1.661223      12
50     5   100 0.462829    0.903286    0.880915    -20.52    1.688331      12
50     5    50 0.449660    0.894819    0.890319    -20.80    1.679530      12
50     5    60 0.448676    0.893417    0.889481    -21.73    1.671993      12
60     3   100 0.447752    0.865484    0.835466    -15.43    1.586763      12
40     5    80 0.425206    0.877317    0.904222    -21.41    1.692305      12
40     5   100 0.423504    0.872672    0.898336    -21.85    1.662718      12
60    